# Rerank lại ứng viên bằng Qwen3-Reranker-4B (bước A2)

**Chuẩn bị:** Settings → Accelerator **GPU T4 x2**, Internet **On**. Add Input: dataset chứa `pairs.parquet`
(xuất bằng `export_rerank_candidates.py`). Nếu đang chạy tiếp lần trước: Add Input thêm output của lần chạy đó
(có `scores.parquet` / `scores_gpu*.jsonl`) — các cặp đã chấm sẽ được bỏ qua.

**Kết quả:** `/kaggle/working/scores.parquet` → tải về, đặt vào `work/rerank/` rồi chạy `build_rerank_submission.py`.

In [ ]:
# ===== CẤU HÌNH =====
MODEL_ID = 'Qwen/Qwen3-Reranker-4B'  # 4B fp16 vừa T4 16GB
TOP_K = 100            # mỗi câu chấm K tài liệu đầu theo điểm cũ (chạy thử nhanh: 60)
PASSAGE_RANKS = 1      # số đoạn mỗi tài liệu được chấm (<= --passages-per-doc lúc xuất)
MAX_LENGTH = 768       # token tối đa của mỗi cặp (prompt + câu hỏi + đoạn)
BATCH_SIZE = 16        # tự giảm khi hết VRAM
SHARD_INDEX, SHARD_COUNT = 0, 1  # chia cho nhiều tài khoản Kaggle: tài khoản i đặt SHARD_INDEX = i
TIME_LIMIT_HOURS = 11.0          # dừng an toàn trước giới hạn 12 giờ của Kaggle
DUMMY = False          # True: chạy thử đường ống, không cần GPU/model

In [ ]:
import glob, json, os, subprocess, sys, threading, time
START = time.time()
WORK = '/kaggle/working'
INPUT = '/kaggle/input'

found = sorted(glob.glob(f'{INPUT}/**/pairs.parquet', recursive=True))
if not found:
    for root, dirs, files in os.walk(INPUT):
        print(' ', root, files[:10])
    raise FileNotFoundError('Chưa gắn dataset chứa pairs.parquet (Add Input).')
PAIRS = found[0]
RESUME = sorted(glob.glob(f'{INPUT}/**/scores.parquet', recursive=True)
                + glob.glob(f'{INPUT}/**/scores_gpu*.jsonl', recursive=True))
print('PAIRS =', PAIRS)
print('Kết quả cũ để bỏ qua:', RESUME or 'không có')

if DUMMY:
    N_GPU = 2
else:
    import torch, transformers
    N_GPU = torch.cuda.device_count()
    print('torch', torch.__version__, '| transformers', transformers.__version__, '| GPU:', N_GPU,
          [torch.cuda.get_device_name(i) for i in range(N_GPU)])
    if N_GPU == 0:
        raise RuntimeError('Không có GPU: Settings -> Accelerator -> GPU T4 x2.')
    major, minor = (int(x) for x in transformers.__version__.split('.')[:2])
    if (major, minor) < (4, 51):
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers>=4.51'], check=True)
        print('Đã nâng transformers (các tiến trình chấm điểm sẽ dùng bản mới).')

In [ ]:
WORKER_SRC = '"""Chấm lại các cặp (câu hỏi, đoạn) bằng Qwen3-Reranker trên MỘT GPU (bước A2, chạy trên Kaggle).\n\nĐọc một file parquet các cặp (qid, query, doc_id, run, chunk_order, text), ghi từng dòng JSON\n{"key": ..., "score": ...} vào --out và flush sau mỗi batch, nên dừng giữa chừng không mất gì; chạy\nlại với cùng --out thì bỏ qua các cặp đã chấm. Prompt và cách tính điểm giống hệt reranker Qwen3 của\npipeline (src/vibiomir/retrieval/rerank.py): xác suất "yes" so với "no" ở token cuối.\n"""\nimport argparse\nimport json\nimport math\nimport os\nimport re\nimport sys\nimport time\n\nimport pyarrow.parquet as pq\n\nINSTRUCTION = (\'Given a Vietnamese medical question, judge whether the document passage \'\n               \'contains information that answers it.\')\nPREFIX = (\'<|im_start|>system\\nJudge whether the Document meets the requirements based on the Query \'\n          \'and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\\n\'\n          \'<|im_start|>user\\n\')\nSUFFIX = \'<|im_end|>\\n<|im_start|>assistant\\n<think>\\n\\n</think>\\n\\n\'\n\n\ndef pair_key(qid, doc_id, run, order):\n    return f\'{int(qid)}:{int(doc_id)}:{run}:{int(order)}\'\n\n\ndef done_keys(path):\n    keys = set()\n    if os.path.exists(path):\n        with open(path, encoding=\'utf-8\') as stream:\n            for line in stream:\n                try:\n                    keys.add(json.loads(line)[\'key\'])\n                except (ValueError, KeyError):\n                    pass  # A line cut by a crash; that pair is simply scored again.\n    return keys\n\n\nclass DummyScorer:\n    """Word overlap, for testing the plumbing without a GPU or a model download."""\n\n    def score(self, queries, passages):\n        out = []\n        for query, passage in zip(queries, passages):\n            q = set(re.findall(r\'\\w+\', query.lower()))\n            p = set(re.findall(r\'\\w+\', passage.lower()))\n            out.append(len(q & p) / max(1, len(q)))\n        return out\n\n\nclass QwenScorer:\n    def __init__(self, model_id, max_length):\n        import torch\n        from transformers import AutoModelForCausalLM, AutoTokenizer\n\n        self.torch = torch\n        self.tokenizer = AutoTokenizer.from_pretrained(model_id, padding_side=\'left\')\n        self.model = AutoModelForCausalLM.from_pretrained(\n            model_id, torch_dtype=torch.float16, attn_implementation=\'sdpa\').to(\'cuda\').eval()\n        self.yes_id = self.tokenizer.convert_tokens_to_ids(\'yes\')\n        self.no_id = self.tokenizer.convert_tokens_to_ids(\'no\')\n        self.prefix_ids = self.tokenizer.encode(PREFIX, add_special_tokens=False)\n        self.suffix_ids = self.tokenizer.encode(SUFFIX, add_special_tokens=False)\n        self.budget = max_length - len(self.prefix_ids) - len(self.suffix_ids)\n\n    def score(self, queries, passages):\n        prompts = [f\'<Instruct>: {INSTRUCTION}\\n<Query>: {q}\\n<Document>: {p}\'\n                   for q, p in zip(queries, passages)]\n        encoded = self.tokenizer(prompts, padding=False, truncation=True, add_special_tokens=False,\n                                 return_attention_mask=False, max_length=self.budget)\n        encoded[\'input_ids\'] = [self.prefix_ids + ids + self.suffix_ids for ids in encoded[\'input_ids\']]\n        inputs = self.tokenizer.pad(encoded, padding=True, return_tensors=\'pt\').to(\'cuda\')\n        with self.torch.inference_mode():\n            last = self.model(**inputs, use_cache=False, logits_to_keep=1).logits[:, -1, :]\n            pair = self.torch.stack([last[:, self.no_id], last[:, self.yes_id]], dim=1).float()\n            return self.torch.softmax(pair, dim=1)[:, 1].cpu().tolist()\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__.splitlines()[0])\n    parser.add_argument(\'--pairs\', required=True)\n    parser.add_argument(\'--out\', required=True)\n    parser.add_argument(\'--model\', default=\'Qwen/Qwen3-Reranker-4B\')\n    parser.add_argument(\'--max-length\', type=int, default=768)\n    parser.add_argument(\'--batch-size\', type=int, default=16)\n    parser.add_argument(\'--deadline\', type=float, default=0, help=\'Unix time to stop at; 0 = none.\')\n    parser.add_argument(\'--tag\', default=\'\')\n    parser.add_argument(\'--dummy\', action=\'store_true\')\n    args = parser.parse_args()\n\n    table = pq.read_table(args.pairs, columns=[\'qid\', \'query\', \'doc_id\', \'run\', \'chunk_order\', \'text\'])\n    rows = list(zip(*(table[name].to_pylist() for name in table.column_names)))\n    done = done_keys(args.out)\n    todo = [row for row in rows if pair_key(row[0], row[2], row[3], row[4]) not in done]\n    # Longest first: an out-of-memory batch shows up at the start and the batch size settles there.\n    todo.sort(key=lambda row: -(len(row[1]) + len(row[5])))\n    print(f\'{args.tag} cặp: {len(rows)}, đã chấm: {len(rows) - len(todo)}, còn: {len(todo)}\', flush=True)\n    if not todo:\n        return\n    scorer = DummyScorer() if args.dummy else QwenScorer(args.model, args.max_length)\n    size, position, written, bad = args.batch_size, 0, 0, 0\n    started = last_report = time.time()\n    with open(args.out, \'a\', encoding=\'utf-8\') as out:\n        while position < len(todo):\n            if args.deadline and time.time() > args.deadline:\n                print(f\'{args.tag} hết giờ, dừng ở {position}/{len(todo)}; chạy lại để làm tiếp.\', flush=True)\n                break\n            batch = todo[position:position + size]\n            try:\n                scores = scorer.score([r[1] for r in batch], [r[5] for r in batch])\n            except RuntimeError as error:\n                if \'out of memory\' not in str(error).lower() or size == 1:\n                    raise\n                size = max(1, size // 2)\n                scorer.torch.cuda.empty_cache()\n                print(f\'{args.tag} hết VRAM, giảm batch còn {size}\', flush=True)\n                continue\n            for row, score in zip(batch, scores):\n                if score is None or not math.isfinite(score):\n                    bad += 1\n                    score = None\n                out.write(json.dumps({\'key\': pair_key(row[0], row[2], row[3], row[4]), \'score\': score}) + \'\\n\')\n            out.flush()\n            position += len(batch)\n            written += len(batch)\n            if written >= 200 and bad > written // 2:\n                raise RuntimeError(f\'{bad}/{written} điểm không hợp lệ (NaN/inf) với fp16. \'\n                                   \'Mô hình này không chạy được fp16 trên GPU hiện tại; đổi MODEL_ID.\')\n            now = time.time()\n            if now - last_report > 60 or position >= len(todo):\n                rate = written / max(1e-9, now - started)\n                eta = (len(todo) - position) / max(rate, 1e-9) / 60\n                print(f\'{args.tag} {position}/{len(todo)} cặp, {rate:.1f} cặp/s, còn ~{eta:.0f} phút, \'\n                      f\'batch {size}, điểm lỗi {bad}\', flush=True)\n                last_report = now\n    print(f\'{args.tag} xong {written} cặp trong {(time.time() - started) / 60:.1f} phút, điểm lỗi {bad}\', flush=True)\n\n\nif __name__ == \'__main__\':\n    sys.stdout.reconfigure(encoding=\'utf-8\')\n    main()\n'
with open(f'{WORK}/rerank_worker.py', 'w', encoding='utf-8') as f:
    f.write(WORKER_SRC)
print('Đã ghi rerank_worker.py')

In [ ]:
import numpy as np
import pyarrow as pa, pyarrow.compute as pc, pyarrow.parquet as pq

table = pq.read_table(PAIRS)
mask = ((table['old_doc_rank'].to_numpy() < TOP_K)
        & (table['passage_rank'].to_numpy() < PASSAGE_RANKS)
        & (table['qid'].to_numpy() % SHARD_COUNT == SHARD_INDEX))
table = table.filter(pa.array(mask))

def key_of(q, d, r, o):
    return f'{int(q)}:{int(d)}:{r}:{int(o)}'

done = set()
for path in RESUME:
    if path.endswith('.parquet'):
        t = pq.read_table(path, columns=['key', 'score'])
        done.update(k for k, s in zip(t['key'].to_pylist(), t['score'].to_pylist()) if s is not None)
    else:
        with open(path, encoding='utf-8') as f:
            for line in f:
                try:
                    row = json.loads(line)
                except ValueError:
                    continue
                if row.get('score') is not None:
                    done.add(row['key'])
keys = [key_of(*r) for r in zip(*(table[c].to_pylist() for c in ('qid', 'doc_id', 'run', 'chunk_order')))]
todo = table.filter(pa.array([k not in done for k in keys]))
print(f'Cặp trong phần việc này: {table.num_rows:,} | đã chấm trước: {table.num_rows - todo.num_rows:,} | còn: {todo.num_rows:,}')

# Chia đều cho các GPU theo độ dài (xếp theo độ dài rồi chia vòng tròn).
lengths = pc.add(pc.utf8_length(todo['text']), pc.utf8_length(todo['query'])).to_pylist()
order = sorted(range(todo.num_rows), key=lambda i: -lengths[i])
PARTS = []
for g in range(N_GPU):
    part = todo.take(pa.array(order[g::N_GPU], type=pa.int64()))
    path = f'{WORK}/part{g}.parquet'
    pq.write_table(part, path)
    PARTS.append(path)
    print(f'GPU{g}: {part.num_rows:,} cặp')

In [ ]:
if not DUMMY and todo.num_rows:
    from huggingface_hub import snapshot_download
    t0 = time.time()
    snapshot_download(MODEL_ID)
    print(f'Đã tải {MODEL_ID} trong {(time.time() - t0) / 60:.1f} phút')

In [ ]:
deadline = START + TIME_LIMIT_HOURS * 3600
procs = []

def pump(proc):
    for line in proc.stdout:
        print(line, end='', flush=True)

for g, part in enumerate(PARTS):
    cmd = [sys.executable, '-u', f'{WORK}/rerank_worker.py', '--pairs', part, '--out', f'{WORK}/scores_gpu{g}.jsonl',
           '--model', MODEL_ID, '--max-length', str(MAX_LENGTH), '--batch-size', str(BATCH_SIZE),
           '--deadline', str(deadline), '--tag', f'[GPU{g}]'] + (['--dummy'] if DUMMY else [])
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g), TOKENIZERS_PARALLELISM='false')
    proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    threading.Thread(target=pump, args=(proc,), daemon=True).start()
    procs.append(proc)
codes = [proc.wait() for proc in procs]
time.sleep(2)
print('Mã thoát:', codes)

In [ ]:
scores = {}
for path in RESUME + sorted(glob.glob(f'{WORK}/scores_gpu*.jsonl')):
    if path.endswith('.parquet'):
        t = pq.read_table(path, columns=['key', 'score'])
        pairs_iter = zip(t['key'].to_pylist(), t['score'].to_pylist())
    else:
        pairs_iter = []
        with open(path, encoding='utf-8') as f:
            for line in f:
                try:
                    row = json.loads(line)
                except ValueError:
                    continue
                pairs_iter.append((row['key'], row['score']))
    for k, s in pairs_iter:
        if s is not None:
            scores[k] = s
pq.write_table(pa.table({'key': list(scores), 'score': pa.array(list(scores.values()), type=pa.float32())}),
               f'{WORK}/scores.parquet')
need = set(keys)
have = sum(1 for k in need if k in scores)
print(f'scores.parquet: {len(scores):,} cặp có điểm | phần việc này: {have:,}/{len(need):,} ({100 * have / max(1, len(need)):.1f}%)')
if have < len(need):
    print('CHƯA XONG: Save Version lại với output này làm Input để chấm nốt.')

# So sánh nhanh thứ hạng cũ và mới
import numpy as np
q = table['qid'].to_pylist(); old = table['old_score'].to_pylist()
new = [scores.get(k) for k in keys]
by_q = {}
for qi, o, n, k in zip(q, old, new, keys):
    if n is not None:
        by_q.setdefault(qi, []).append((o, n, k.split(':')[1]))
overlap = []
for items in by_q.values():
    top_old = {d for _o, _n, d in sorted(items, key=lambda x: -x[0])[:60]}
    top_new = {d for _o, _n, d in sorted(items, key=lambda x: -x[1])[:60]}
    overlap.append(len(top_old & top_new) / max(1, min(60, len(items))))
if overlap:
    print(f'Trùng nhau giữa top-60 cũ và top-60 mới: trung bình {np.mean(overlap):.0%}')
    vals = [n for items in by_q.values() for _o, n, _d in items]
    print('Phân vị điểm mới:', {p: round(float(np.percentile(vals, p)), 3) for p in (10, 25, 50, 75, 90)})
print(f'Tổng thời gian: {(time.time() - START) / 60:.1f} phút')